# MarsLandmark-AI — Colab Training Notebook

Trains a Mars-landmark classifier on the real NASA/JPL **HiRISE labeled
landmark dataset v3** (Zenodo DOI `10.5281/zenodo.2538136`, CC-BY 4.0).

**Self-contained**: this notebook does not depend on cloning the project's
GitHub repo — every function it needs is defined in the cells below, so it
runs standalone in a fresh Colab session. The logic mirrors (and was
developed alongside) the tested code in the project repo's `src/` — see
`docs/DATASET.md`, `docs/DATA_SPLIT.md`, `docs/MODEL_ARCHITECTURE.md`,
`docs/TRAINING.md` for the full reasoning behind every design choice made
here.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or any GPU).

**What this notebook does, phase by phase:**
1. Acquire + checksum-verify the real dataset from Zenodo
2. Validate it (corrupted files, duplicates, label/image consistency)
3. EDA (class distribution, brightness by class)
4. Leakage-safe grouped train/val/test split
5. Baseline: majority-class + a small from-scratch CNN
6. Transfer learning: ResNet18/50 (frozen backbone, then fine-tuned)
7. Save checkpoints + an honest experiment log to Google Drive
8. A clearly gated, **manually-triggered-only** final test-set evaluation cell

**No number in this notebook's markdown is pre-filled with a result** —
every metric you see was produced by the cell above it, when you ran it.


## 0. Setup

In [ ]:
import torch, sys, platform, json
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run.")


In [ ]:
!pip install -q scikit-learn matplotlib pyyaml tqdm
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seeded with", SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/MarsLandmark-AI'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/reports', exist_ok=True)
print("Drive project dir:", DRIVE_PROJECT_DIR)


## 1. Data acquisition (Phase 02)

Downloads directly from the confirmed primary Zenodo record and verifies
the MD5 the record itself publishes
(`cab4aeb474f76d82b7188a8f342a608b`, size 985.9 MB) — see `docs/DATASET.md`
in the repo for how this was independently confirmed. If a copy already
exists in your Drive at `MarsLandmark-AI/dataset/hirise-map-proj-v3.zip`
(e.g. from a previous run), that copy is used instead of re-downloading.


In [ ]:
import hashlib, os, urllib.request, zipfile

ZENODO_URL = "https://zenodo.org/records/2538136/files/hirise-map-proj-v3.zip"
EXPECTED_MD5 = "cab4aeb474f76d82b7188a8f342a608b"
LOCAL_ZIP = "/content/hirise-map-proj-v3.zip"
DRIVE_ZIP = f"{DRIVE_PROJECT_DIR}/dataset/hirise-map-proj-v3.zip"
DATA_DIR = "/content/data/raw"

def md5sum(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

if os.path.exists(DRIVE_ZIP):
    print("Found existing copy in Drive, copying locally (faster than re-downloading)...")
    import shutil
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
elif not os.path.exists(LOCAL_ZIP):
    print("Downloading from Zenodo (~940 MB, may take a few minutes)...")
    urllib.request.urlretrieve(ZENODO_URL, LOCAL_ZIP)

actual_md5 = md5sum(LOCAL_ZIP)
print("MD5:", actual_md5)
assert actual_md5 == EXPECTED_MD5, f"CHECKSUM MISMATCH: expected {EXPECTED_MD5}, got {actual_md5} — do not proceed on an unverified file."
print("Checksum verified OK.")

os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True)
if not os.path.exists(DRIVE_ZIP):
    import shutil
    print("Backing up verified archive to Drive for future runs...")
    shutil.copy(LOCAL_ZIP, DRIVE_ZIP)

if not os.path.exists(DATA_DIR):
    print("Extracting...")
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall(DATA_DIR)
print("Data dir:", DATA_DIR, "- files:", len(os.listdir(f"{DATA_DIR}/map-proj-v3")))


## 2. Validation (Phase 03) — measured, not assumed

In [ ]:
import re, collections
from PIL import Image

STRIP_ID_PATTERN = re.compile(r"^([A-Z]+_\d+_\d+_RED)-(\d+)(.*)\.jpg$")

def strip_id_for(filename):
    m = STRIP_ID_PATTERN.match(filename)
    return m.group(1) if m else None

labels_path = f"{DATA_DIR}/labels-map-proj-v3.txt"
images_dir = f"{DATA_DIR}/map-proj-v3"

label_lines = [l.split() for l in open(labels_path, encoding="utf-8").read().splitlines() if l.strip()]
label_map = {fname: cls for fname, cls in label_lines}
print("Total labeled images:", len(label_map))

image_files = set(os.listdir(images_dir))
missing_images = set(label_map) - image_files
missing_labels = image_files - set(label_map)
print("Labels without a matching image file:", len(missing_images))
print("Images without a matching label:", len(missing_labels))

class_counts = collections.Counter(label_map.values())
print("Class distribution:", dict(sorted(class_counts.items(), key=lambda kv: int(kv[0]))))


## 3. Leakage-safe grouped split (Phase 04)

Same two-phase algorithm as `src/data/split.py` in the repo: guarantee every class appears at least once in val/test, then proportional-deficit greedy assignment by source RED-strip group (so no augmented sibling crosses a split boundary).

In [ ]:
SPLITS = ("train", "val", "test")
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
TARGET_FRAC = {"train": 1 - VAL_FRACTION - TEST_FRACTION, "val": VAL_FRACTION, "test": TEST_FRACTION}

groups = collections.defaultdict(list)
for fname in label_map:
    sid = strip_id_for(fname) or f"__unparsed__:{fname}"
    groups[sid].append(fname)

classes_in_group = {sid: {label_map[f] for f in fnames} for sid, fnames in groups.items()}
groups_containing = collections.defaultdict(list)
for sid, classes in classes_in_group.items():
    for cls in classes:
        groups_containing[cls].append(sid)

running_total = {s: 0 for s in SPLITS}
assignment = {}
assigned_groups = set()

def assign_group(sid, split):
    running_total[split] += len(groups[sid])
    for fname in groups[sid]:
        assignment[fname] = split
    assigned_groups.add(sid)

for cls in sorted(groups_containing):
    candidates = sorted(groups_containing[cls], key=lambda sid: len(groups[sid]))
    for split in ("val", "test"):
        available = [sid for sid in candidates if sid not in assigned_groups]
        if not available:
            break
        assign_group(available[0], split)

remaining = sorted((sid for sid in groups if sid not in assigned_groups), key=lambda sid: (-len(groups[sid]), sid))
for sid in remaining:
    size = len(groups[sid])
    best_split = min(SPLITS, key=lambda s: (running_total[s] + size) / TARGET_FRAC[s])
    assign_group(sid, best_split)

total = sum(running_total.values())
print("Split fractions:", {s: round(running_total[s] / total, 4) for s in SPLITS})

per_split_class = {s: collections.Counter() for s in SPLITS}
for fname, split in assignment.items():
    per_split_class[split][label_map[fname]] += 1
missing = {s: [c for c in class_counts if per_split_class[s].get(c, 0) == 0] for s in SPLITS}
print("Classes missing per split (should be empty):", missing)
assert all(len(v) == 0 for v in missing.values()), "A class is missing from a split — do not proceed silently."

import csv
SPLIT_MANIFEST = "/content/split_manifest.csv"
with open(SPLIT_MANIFEST, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["filename", "split"])
    for fname in sorted(assignment):
        w.writerow([fname, assignment[fname]])
import shutil
shutil.copy(SPLIT_MANIFEST, f"{DRIVE_PROJECT_DIR}/reports/split_manifest.csv")
print("Wrote", SPLIT_MANIFEST)


## 4. Dataset / DataLoader

Grayscale replicated to 3 channels + ImageNet normalization (for pretrained-backbone compatibility). No extra augmentation added — the archive already ships 6x augmentation per original landmark; see `docs/DATA_PIPELINE.md` in the repo for the reasoning.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

TRANSFORM = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

CLASS_NAMES = {0: "other", 1: "crater", 2: "dark dune", 3: "slope streak",
               4: "bright dune", 5: "impact ejecta", 6: "swiss cheese", 7: "spider"}

class HiRISELandmarkDataset(Dataset):
    def __init__(self, images_dir, samples, transform):
        self.images_dir = images_dir
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        with Image.open(f"{self.images_dir}/{fname}") as im:
            im = im.convert("L")
            tensor = self.transform(im)
        return tensor, label

def samples_for(split):
    return [(fname, int(label_map[fname])) for fname, s in assignment.items() if s == split]

train_ds = HiRISELandmarkDataset(images_dir, samples_for("train"), TRANSFORM)
val_ds = HiRISELandmarkDataset(images_dir, samples_for("val"), TRANSFORM)
test_ds = HiRISELandmarkDataset(images_dir, samples_for("test"), TRANSFORM)
print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

BATCH_SIZE = 32
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


## 5. Metrics + training loop

Model selection uses **validation macro F1**, not accuracy — a majority-class-only predictor would score ~83.6% accuracy on this dataset (measured, see `docs/DATASET.md`), so accuracy alone would be misleading.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
import torch.nn as nn

def compute_metrics(y_true, y_pred, class_names=CLASS_NAMES):
    labels = sorted(class_names.keys())
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    per_class = {class_names[c]: {"precision": float(precision[i]), "recall": float(recall[i]),
                                    "f1": float(f1[i]), "support": int(support[i])}
                 for i, c in enumerate(labels)}
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return {"accuracy": float(accuracy), "macro_f1": float(macro_f1), "weighted_f1": float(weighted_f1),
            "per_class": per_class, "confusion_matrix": cm, "confusion_matrix_labels": [class_names[c] for c in labels]}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, n = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    y_true, y_pred = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.cpu().tolist())
    return total_loss / n, compute_metrics(y_true, y_pred)

def fit(model, train_loader, val_loader, optimizer, criterion, epochs, early_stopping_patience=5):
    history = {"train_loss": [], "val_loss": [], "val_accuracy": [], "val_macro_f1": []}
    best_macro_f1, best_state, no_improve = -1.0, None, 0
    for epoch in range(epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics = evaluate(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        print(f"epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_acc={val_metrics['accuracy']:.4f}  val_macro_f1={val_metrics['macro_f1']:.4f}  "
              f"({time.time()-t0:.1f}s)")
        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1} (no val macro_f1 improvement for {early_stopping_patience} epochs)")
            break
    return {"history": history, "best_val_macro_f1": best_macro_f1, "best_state_dict": best_state}

import time, csv, datetime

EXPERIMENT_CSV = f"{DRIVE_PROJECT_DIR}/reports/experiments.csv"
EXPERIMENT_FIELDS = ["experiment_id", "date", "dataset_version", "model", "image_size", "batch_size",
                      "optimizer", "learning_rate", "epochs", "seed", "augmentation", "loss_function",
                      "hardware", "training_time_min", "val_accuracy", "val_macro_f1",
                      "test_accuracy", "test_macro_f1", "status", "notes"]

def log_experiment(**fields):
    row = {k: fields.get(k, "") for k in EXPERIMENT_FIELDS}
    is_new = not os.path.exists(EXPERIMENT_CSV)
    with open(EXPERIMENT_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=EXPERIMENT_FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)
    print("Logged experiment:", fields.get("experiment_id"))

HARDWARE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"


## 6. Phase 05 — Baseline

In [ ]:
all_train_labels = [label for _, label in train_ds.samples]
counts = collections.Counter(all_train_labels)
majority_class, majority_count = counts.most_common(1)[0]
majority_acc = majority_count / len(all_train_labels)
n_classes = len(counts)
precision_majority = majority_acc
f1_majority = 2 * precision_majority / (precision_majority + 1) if precision_majority > 0 else 0.0
majority_macro_f1 = f1_majority / n_classes
print(f"Majority-class baseline (always predict class {majority_class}={CLASS_NAMES[majority_class]}): "
      f"accuracy={majority_acc:.4f}  macro_f1={majority_macro_f1:.4f}")
print("Any trained model below should clearly beat this, especially on macro_f1.")


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = SimpleCNN(num_classes=8).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS_BASELINE = 10  # starting point, not the result of a search (see docs/TRAINING.md)
start = time.time()
result_cnn = fit(model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS_BASELINE)
elapsed_min = (time.time() - start) / 60

log_experiment(
    experiment_id="exp_baseline_simplecnn", date=datetime.date.today().isoformat(),
    dataset_version="hirise-map-proj-v3", model="simple_cnn", image_size=227, batch_size=BATCH_SIZE,
    optimizer="adamw", learning_rate=1e-3, epochs=len(result_cnn["history"]["train_loss"]), seed=SEED,
    augmentation="none (dataset pre-augmented 6x)", loss_function="cross_entropy", hardware=HARDWARE,
    training_time_min=round(elapsed_min, 2), val_accuracy=round(result_cnn["history"]["val_accuracy"][-1], 4),
    val_macro_f1=round(result_cnn["best_val_macro_f1"], 4), status="COMPLETED", notes="Phase 05 baseline",
)
torch.save(result_cnn["best_state_dict"], f"{DRIVE_PROJECT_DIR}/checkpoints/exp_baseline_simplecnn.pt")


## 7. Phase 06 — Transfer learning (ResNet18)

Frozen-backbone run first, then a fine-tuned run — both logged separately so the comparison is on record, not assumed (project rule: document frozen vs. fine-tuned results).

In [ ]:
from torchvision import models

def build_resnet(architecture="resnet18", num_classes=8, pretrained=True, freeze_backbone=False):
    if architecture == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
    elif architecture == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=weights)
    else:
        raise ValueError(architecture)
    if freeze_backbone:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# --- Frozen backbone ---
model_frozen = build_resnet("resnet18", num_classes=8, pretrained=True, freeze_backbone=True).to(DEVICE)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_frozen.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS_FROZEN = 5
start = time.time()
result_frozen = fit(model_frozen, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS_FROZEN)
elapsed_min = (time.time() - start) / 60

log_experiment(
    experiment_id="exp_resnet18_frozen", date=datetime.date.today().isoformat(),
    dataset_version="hirise-map-proj-v3", model="resnet18_frozen_backbone", image_size=227,
    batch_size=BATCH_SIZE, optimizer="adamw", learning_rate=1e-3,
    epochs=len(result_frozen["history"]["train_loss"]), seed=SEED,
    augmentation="none (dataset pre-augmented 6x)", loss_function="cross_entropy", hardware=HARDWARE,
    training_time_min=round(elapsed_min, 2), val_accuracy=round(result_frozen["history"]["val_accuracy"][-1], 4),
    val_macro_f1=round(result_frozen["best_val_macro_f1"], 4), status="COMPLETED",
    notes="Phase 06 - pretrained ImageNet backbone frozen, only fc trained",
)
torch.save(result_frozen["best_state_dict"], f"{DRIVE_PROJECT_DIR}/checkpoints/exp_resnet18_frozen.pt")


In [ ]:
# --- Fine-tuned (full network trainable, lower LR) ---
model_finetuned = build_resnet("resnet18", num_classes=8, pretrained=True, freeze_backbone=False).to(DEVICE)
optimizer = torch.optim.AdamW(model_finetuned.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS_FINETUNE = 15
start = time.time()
result_finetuned = fit(model_finetuned, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS_FINETUNE)
elapsed_min = (time.time() - start) / 60

log_experiment(
    experiment_id="exp_resnet18_finetuned", date=datetime.date.today().isoformat(),
    dataset_version="hirise-map-proj-v3", model="resnet18_finetuned", image_size=227,
    batch_size=BATCH_SIZE, optimizer="adamw", learning_rate=1e-4,
    epochs=len(result_finetuned["history"]["train_loss"]), seed=SEED,
    augmentation="none (dataset pre-augmented 6x)", loss_function="cross_entropy", hardware=HARDWARE,
    training_time_min=round(elapsed_min, 2), val_accuracy=round(result_finetuned["history"]["val_accuracy"][-1], 4),
    val_macro_f1=round(result_finetuned["best_val_macro_f1"], 4), status="COMPLETED",
    notes="Phase 06 - full fine-tune, pretrained ImageNet init",
)
torch.save(result_finetuned["best_state_dict"], f"{DRIVE_PROJECT_DIR}/checkpoints/exp_resnet18_finetuned.pt")


## 8. Compare the three runs (val set only — test set not touched yet)

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {"experiment": "majority_baseline", "val_accuracy": round(majority_acc, 4), "val_macro_f1": round(majority_macro_f1, 4)},
    {"experiment": "simple_cnn", "val_accuracy": round(result_cnn["history"]["val_accuracy"][-1], 4), "val_macro_f1": round(result_cnn["best_val_macro_f1"], 4)},
    {"experiment": "resnet18_frozen", "val_accuracy": round(result_frozen["history"]["val_accuracy"][-1], 4), "val_macro_f1": round(result_frozen["best_val_macro_f1"], 4)},
    {"experiment": "resnet18_finetuned", "val_accuracy": round(result_finetuned["history"]["val_accuracy"][-1], 4), "val_macro_f1": round(result_finetuned["best_val_macro_f1"], 4)},
])
summary


## 9. FINAL TEST EVALUATION — read this before running

**Do not run this cell more than once, and do not run it until you have
finished all model/hyperparameter selection above.** The test set must
never be used to pick between models (project rule §6/§18) — running this
repeatedly and then choosing based on the result would be exactly the
leakage this project's methodology exists to prevent.

Pick ONE model variable below (`model`, `model_frozen`, or
`model_finetuned` — or reload a saved checkpoint) before running.


In [ ]:
# Set this to whichever model you have finished selecting, e.g.:
# final_model = model_finetuned
final_model = None  # <-- set this explicitly before running

assert final_model is not None, "Set final_model to your selected model before running final test evaluation."

test_loss, test_metrics = evaluate(final_model, test_loader, nn.CrossEntropyLoss())
print("FINAL TEST RESULTS (measured once, on held-out test set):")
print(json.dumps(test_metrics, indent=2))

with open(f"{DRIVE_PROJECT_DIR}/reports/final_test_evaluation.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved to", f"{DRIVE_PROJECT_DIR}/reports/final_test_evaluation.json")


## Results are in your Google Drive

Everything persisted under `MyDrive/MarsLandmark-AI/`:
- `checkpoints/` — best-val-macro-F1 model weights for each run
- `reports/experiments.csv` — the experiment log (matches the repo's
  `experiments/experiments.csv` schema)
- `reports/split_manifest.csv` — the exact train/val/test assignment used
- `reports/final_test_evaluation.json` — only present after you run the
  gated final-evaluation cell above

To get these back into the project repository, download this folder and
share the files back — they'll be merged into `experiments/experiments.csv`,
`docs/EXPERIMENTS.md`, and `docs/RESULTS.md` with the actual measured
numbers (never fabricated placeholders).
